**Como está a performance de execução dos serviços técnicos**

In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS workspace.silver;

CREATE OR REPLACE TABLE workspace.silver.technical_service AS
SELECT
    id,
    fk_technical_project,
    purpose,
    status,
    created_at,
    end_date
FROM solaria.public.technical_service
WHERE fk_technical_project IS NOT NULL
QUALIFY ROW_NUMBER() OVER (PARTITION BY id ORDER BY created_at DESC) = 1;

In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS workspace.gold;

CREATE OR REPLACE TABLE workspace.gold.dim_tempo_servico AS
SELECT DISTINCT
    CAST(date_trunc('day', created_at) AS DATE) AS data_id,
    YEAR(created_at)  AS ano,
    MONTH(created_at) AS mes,
    QUARTER(created_at) AS trimestre
FROM workspace.silver.technical_service;

CREATE OR REPLACE TABLE workspace.gold.dim_tecnico AS
SELECT DISTINCT
    ta.id AS technician_affiliation_id,
    t.id  AS technician_id,
    p.name AS nome,
    c.trade_name AS empresa,
    ta.affiliation_type
FROM solaria.public.technician_affiliation ta
JOIN solaria.public.technician t ON t.id = ta.fk_technician
JOIN solaria.public.person p ON p.id = t.fk_person
LEFT JOIN solaria.public.company c ON c.id = ta.fk_company;

CREATE OR REPLACE TABLE workspace.gold.fato_servicos AS
SELECT
    ts.id AS service_id,
    ts.fk_technical_project AS project_id,
    se.fk_technician_affiliation AS technician_affiliation_id,
    ts.status,
    CAST(date_trunc('day', ts.created_at) AS DATE) AS data_id,
    ts.created_at,
    ts.end_date,
    sc.delivery_deadline
FROM workspace.silver.technical_service ts
LEFT JOIN solaria.public.service_executor se ON se.fk_service = ts.id
LEFT JOIN solaria.public.service_contract sc ON sc.fk_service = ts.id;

In [0]:
%sql
CREATE OR REPLACE TABLE workspace.gold.kpi_tempo_conclusao AS
SELECT
    AVG(DATEDIFF(day, created_at, end_date)) AS dias_medio_conclusao,
    STDDEV(DATEDIFF(day, created_at, end_date)) AS desvio_dias_conclusao
FROM workspace.gold.fato_servicos
WHERE status = 'COMPLETED' AND end_date IS NOT NULL;

CREATE OR REPLACE TABLE workspace.gold.kpi_taxa_no_prazo AS
SELECT
    try_divide(COUNT(*) FILTER (WHERE end_date <= delivery_deadline) * 100.0, COUNT(*)) AS percentual_no_prazo,
    COUNT(*) AS total_avaliados
FROM workspace.gold.fato_servicos
WHERE status = 'COMPLETED' AND end_date IS NOT NULL AND delivery_deadline IS NOT NULL;

CREATE OR REPLACE TABLE workspace.gold.kpi_servicos_por_tecnico AS
SELECT
    dt.nome,
    dt.empresa,
    COUNT(*) AS total_servicos,
    RANK() OVER (ORDER BY COUNT(*) DESC) AS ranking
FROM workspace.gold.fato_servicos f
JOIN workspace.gold.dim_tecnico dt ON dt.technician_affiliation_id = f.technician_affiliation_id
GROUP BY dt.nome, dt.empresa;